# Task 2 — Professional Data Cleaning
## Oasis Infobyte Internship

**Name:**   Mehwish Iqbal 

Task: Task 2






## Objective
Demonstrate professional-level data cleaning by taking a deliberately messy employee dataset and systematically transforming it into a clean, analysis-ready dataset.

### Internship Task Checklist
- Load the dataset and produce a data quality report
- Handle missing data with justified strategies
- Identify and remove duplicate rows
- Standardize inconsistent formatting
- Detect and document outliers using the IQR method
- Correct data types (dates, IDs, monetary values, etc.)
- Produce a final quality report and export the cleaned dataset

**Tools:** Python, pandas, NumPy, Jupyter Notebook


In [96]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")


Libraries imported successfully.


## 1. Load Dataset

The raw CSV file is loaded without modifying the original data. Keeping the raw data unchanged makes the cleaning process reproducible and auditable.


In [97]:
# Load the raw dataset
file_path = "Messy_Employee_dataset.csv"
df = pd.read_csv(file_path)

print(f"Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


Dataset loaded successfully: 1,020 rows × 12 columns


,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.00,DevOps-California,Active,4/2/2021,"59,767.65",bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,"65,304.66",bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,"88,145.90",alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.00,Admin-Nevada,Inactive,11/27/2021,"69,450.99",eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.00,Cloud Tech-Florida,Active,1/5/2022,"109,324.61",frank.williams@example.com,-1586734256,Poor,False


In [98]:
# Basic dataset inspection
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Count"))

print("\nDuplicate rows:", df.duplicated().sum())


Shape: (1020, 12)

Column names:
['Employee_ID', 'First_Name', 'Last_Name', 'Age', 'Department_Region', 'Status', 'Join_Date', 'Salary', 'Email', 'Phone', 'Performance_Score', 'Remote_Work']

Data types:


,Data Type
Employee_ID,object
First_Name,object
Last_Name,object
Age,float64
Department_Region,object
Status,object
Join_Date,object
Salary,float64
Email,object
Phone,int64



Missing values:


,Missing Count
Employee_ID,0
First_Name,0
Last_Name,0
Age,211
Department_Region,0
Status,0
Join_Date,0
Salary,24
Email,0
Phone,0



Duplicate rows: 0


## 2. Initial Data Quality Report

A data quality report checks four important areas:

1. Missing values
2. Duplicate rows
3. Data type issues
4. Value/range anomalies

This report is created **before cleaning** so that improvements can be measured later.


In [99]:
# Initial data quality report
quality_report = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Missing_%": (df.isna().mean() * 100).round(2),
    "Unique_Values": df.nunique(dropna=True)
})

display(quality_report)


,Data_Type,Missing_Count,Missing_%,Unique_Values
Employee_ID,object,0,0.00,1020
First_Name,object,0,0.00,8
Last_Name,object,0,0.00,8
Age,float64,211,20.69,4
Department_Region,object,0,0.00,36
Status,object,0,0.00,3
Join_Date,object,0,0.00,760
Salary,float64,24,2.35,978
Email,object,0,0.00,64
Phone,int64,0,0.00,1020


### Range and validity checks

For this dataset:
- Age should be within a reasonable working-age range.
- Salary should be positive.
- Employee IDs should follow an employee-ID pattern.
- Emails should have a basic valid email structure.
- Join dates should be parseable as dates.
- Phone numbers should be treated as identifiers rather than numerical measures.


In [100]:
# Value range / validity checks
age_invalid = ((df["Age"] < 18) | (df["Age"] > 65)).sum()
salary_invalid = (df["Salary"] <= 0).sum()
id_invalid = (~df["Employee_ID"].astype(str).str.match(r"^EMP\d+$")).sum()
email_invalid = (~df["Email"].astype(str).str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")).sum()
date_invalid = pd.to_datetime(df["Join_Date"], errors="coerce").isna().sum()
phone_length_distribution = df["Phone"].astype(str).str.len().value_counts().sort_index()

range_checks = pd.DataFrame({
    "Check": [
        "Age outside 18–65",
        "Salary <= 0",
        "Invalid Employee_ID pattern",
        "Invalid email format",
        "Unparseable Join_Date"
    ],
    "Issues_Found": [
        age_invalid, salary_invalid, id_invalid, email_invalid, date_invalid
    ]
})

display(range_checks)

print("Phone digit-length distribution:")
display(phone_length_distribution.to_frame("Number of Records"))


,Check,Issues_Found
0,Age outside 18–65,0
1,Salary <= 0,0
2,Invalid Employee_ID pattern,0
3,Invalid email format,0
4,Unparseable Join_Date,0


Phone digit-length distribution:


,Number of Records
Phone,
8,2
9,12
10,78
11,928


## 3. Missing Data Handling

Only columns with missing values require imputation or removal.

### Decision rules
- **Age:** Use the median because age is numeric and the median is less sensitive to extreme values than the mean.
- **Salary:** Use the median because salary distributions can be skewed and the median provides a robust central value.
- **Rows:** Do not delete complete employee records just because Age or Salary is missing; valuable information in the other columns would otherwise be lost.

These choices are documented rather than applying one blanket rule to every column.


In [101]:
# Missing-value counts before treatment
missing_before = df.isna().sum()
display(missing_before[missing_before > 0].to_frame("Missing Before"))


,Missing Before
Age,211
Salary,24


In [102]:
# Apply column-specific missing-data strategies
age_median = df["Age"].median()
salary_median = df["Salary"].median()

df["Age"] = df["Age"].fillna(age_median)
df["Salary"] = df["Salary"].fillna(salary_median)

print(f"Age missing values filled with median: {age_median:.0f}")
print(f"Salary missing values filled with median: {salary_median:,.2f}")

print("\nRemaining missing values:")
display(df.isna().sum().to_frame("Missing After"))


Age missing values filled with median: 30
Salary missing values filled with median: 85,547.87

Remaining missing values:


,Missing After
Employee_ID,0
First_Name,0
Last_Name,0
Age,0
Department_Region,0
Status,0
Join_Date,0
Salary,0
Email,0
Phone,0


## 4. Duplicate Removal

Duplicate rows can cause double-counting and distort analysis. We first count duplicates, then remove them.

In this dataset, the duplicate check is performed across the complete row.


In [103]:
# Identify and remove duplicate rows
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
duplicates_removed = duplicates_before - df.duplicated().sum()

print("Duplicates before cleaning:", duplicates_before)
print("Duplicates removed:", duplicates_removed)
print("Rows after duplicate removal:", len(df))


Duplicates before cleaning: 0
Duplicates removed: 0
Rows after duplicate removal: 1020


## 5. Standardization of Text Fields

Text fields should have consistent formatting so that values such as `male`, `Male`, and ` M ` would not be treated as different categories.

The dataset's categorical values are normalized using trimming and consistent case/mapping rules.


In [104]:
# Standardize text columns
text_columns = [
    "Employee_ID", "First_Name", "Last_Name",
    "Department_Region", "Status", "Email", "Performance_Score"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# Standardize categorical fields
df["Status"] = df["Status"].str.title()

performance_map = {
    "poor": "Poor",
    "average": "Average",
    "good": "Good",
    "excellent": "Excellent"
}
df["Performance_Score"] = (
    df["Performance_Score"]
    .str.lower()
    .map(performance_map)
    .fillna(df["Performance_Score"])
)

# Standardize names
df["First_Name"] = df["First_Name"].str.title()
df["Last_Name"] = df["Last_Name"].str.title()

# Email standardization
df["Email"] = df["Email"].str.lower()

print("Standardization completed.")
print("\nStatus values:", sorted(df["Status"].unique()))
print("Performance values:", sorted(df["Performance_Score"].unique()))


Standardization completed.

Status values: ['Active', 'Inactive', 'Pending']
Performance values: ['Average', 'Excellent', 'Good', 'Poor']


### Department and Region standardization

`Department_Region` combines two concepts in one field (for example, `Sales-Texas`). Splitting it creates cleaner analysis fields while retaining the original combined field for traceability.


In [105]:
# Split Department_Region into separate standardized fields
split_values = df["Department_Region"].str.split("-", n=1, expand=True)

df["Department"] = split_values[0].str.strip().str.title()
df["Region"] = split_values[1].str.strip().str.title()

print("Departments:", sorted(df["Department"].unique()))
print("Regions:", sorted(df["Region"].unique()))
display(df[["Department_Region", "Department", "Region"]].head())


Departments: ['Admin', 'Cloud Tech', 'Devops', 'Finance', 'Hr', 'Sales']
Regions: ['California', 'Florida', 'Illinois', 'Nevada', 'New York', 'Texas']


,Department_Region,Department,Region
0,DevOps-California,Devops,California
1,Finance-Texas,Finance,Texas
2,Admin-Nevada,Admin,Nevada
3,Admin-Nevada,Admin,Nevada
4,Cloud Tech-Florida,Cloud Tech,Florida


## 6. Date Standardization

Dates should be stored as true datetime values rather than text. This enables reliable sorting, filtering, date arithmetic, and time-based analysis.

The original `Join_Date` column is converted with `errors="coerce"` so any invalid date becomes missing and can be investigated.


In [106]:
# Convert Join_Date from text to datetime
df["Join_Date"] = pd.to_datetime(df["Join_Date"], errors="coerce")

date_conversion_issues = df["Join_Date"].isna().sum()
print("Date conversion issues:", date_conversion_issues)
print("Join_Date dtype:", df["Join_Date"].dtype)
display(df[["Employee_ID", "Join_Date"]].head())


Date conversion issues: 0
Join_Date dtype: datetime64[ns]


,Employee_ID,Join_Date
0,EMP1000,2021-04-02
1,EMP1001,2020-07-10
2,EMP1002,2023-12-07
3,EMP1003,2021-11-27
4,EMP1004,2022-01-05


## 7. Data Type Correction

IDs and phone numbers are identifiers, not quantities for mathematical calculations. They should therefore be stored as strings.

- `Employee_ID` → string
- `Phone` → string
- `Age` → numeric integer
- `Salary` → numeric float
- `Join_Date` → datetime
- `Remote_Work` → boolean


In [107]:
# Correct data types
df["Employee_ID"] = df["Employee_ID"].astype("string")

# The raw phone column was loaded as an integer. Convert it to string
# and remove a leading negative sign when present as a formatting artifact.
df["Phone"] = df["Phone"].astype("string").str.replace(r"^-", "", regex=True)

df["Age"] = pd.to_numeric(df["Age"], errors="coerce").round().astype("Int64")
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce").astype(float)
df["Performance_Score"] = df["Performance_Score"].astype("string")
df["Remote_Work"] = df["Remote_Work"].astype("boolean")

print(df.dtypes)


Employee_ID          string[python]
First_Name                   object
Last_Name                    object
Age                           Int64
Department_Region            object
Status                       object
Join_Date            datetime64[ns]
Salary                      float64
Email                        object
Phone                string[python]
Performance_Score    string[python]
Remote_Work                 boolean
Department                   object
Region                       object
dtype: object


### Phone-number validation

Phone numbers are not mathematically imputed because changing an unknown contact number would create false information. Instead, they are stored as strings and validated by digit length.

Records with unusual lengths are flagged for business/source verification rather than silently inventing digits.


In [108]:
# Validate phone length without inventing missing digits
df["Phone_Length"] = df["Phone"].str.len()

valid_phone_lengths = [10, 11]
df["Phone_Length_Status"] = np.where(
    df["Phone_Length"].isin(valid_phone_lengths),
    "Valid length",
    "Review"
)

print("Phone validation summary:")
display(df["Phone_Length_Status"].value_counts())

print("\nRows requiring phone review:", (df["Phone_Length_Status"] == "Review").sum())


Phone validation summary:


Phone_Length_Status
Valid length    928
Review           92
Name: count, dtype: int64


Rows requiring phone review: 92


## 8. Outlier Detection Using IQR

The Interquartile Range (IQR) method is used for numeric columns.

**Formula:**
- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Values outside these bounds are potential outliers.

We evaluate **Age** and **Salary** because they are the main continuous numeric measures suitable for this check.


In [109]:
# IQR outlier detection
numeric_for_outliers = ["Age", "Salary"]
outlier_summary = []

for col in numeric_for_outliers:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(count)
    })

outlier_summary = pd.DataFrame(outlier_summary)
display(outlier_summary)


,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count
0,Age,30.00,35.00,5.00,22.50,42.50,0
1,Salary,"68,811.23","100,372.66","31,561.43","21,469.09","147,714.81",0


### Outlier treatment decision

The IQR check does not identify any Age or Salary values outside the calculated bounds.

**Decision:** No values are removed or capped. This is important because outlier treatment should be evidence-based; modifying valid observations without detecting an actual anomaly can damage the dataset.

If future data contains genuine outliers, the correct action should depend on whether they are data-entry errors, legitimate rare cases, or values requiring business review.


In [110]:
# Confirm that no IQR-based outlier treatment is required
print("No Age or Salary values were removed/capped because the IQR method found no outliers.")


No Age or Salary values were removed/capped because the IQR method found no outliers.


## 9. Before vs After Cleaning

A professional cleaning workflow should demonstrate what changed.


In [111]:
# Re-read raw data for an objective before/after comparison
raw = pd.read_csv(file_path)

comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Age",
        "Missing Salary",
        "Join_Date Data Type",
        "Salary Data Type",
        "Employee_ID Data Type",
        "Phone Data Type"
    ],
    "Before": [
        len(raw),
        raw.shape[1],
        int(raw.duplicated().sum()),
        int(raw["Age"].isna().sum()),
        int(raw["Salary"].isna().sum()),
        str(raw["Join_Date"].dtype),
        str(raw["Salary"].dtype),
        str(raw["Employee_ID"].dtype),
        str(raw["Phone"].dtype)
    ],
    "After": [
        len(df),
        df.shape[1],
        int(df.duplicated().sum()),
        int(df["Age"].isna().sum()),
        int(df["Salary"].isna().sum()),
        str(df["Join_Date"].dtype),
        str(df["Salary"].dtype),
        str(df["Employee_ID"].dtype),
        str(df["Phone"].dtype)
    ]
})

display(comparison)


,Metric,Before,After
0,Rows,1020,1020
1,Columns,12,16
2,Duplicate Rows,0,0
3,Missing Age,211,0
4,Missing Salary,24,0
5,Join_Date Data Type,object,datetime64[ns]
6,Salary Data Type,float64,float64
7,Employee_ID Data Type,object,string
8,Phone Data Type,int64,string


## 10. Final Data Quality Report

The final report confirms that missing values and duplicates have been addressed and that the main analytical fields have appropriate data types.


In [112]:
# Final quality report
final_report = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Missing_%": (df.isna().mean() * 100).round(2),
    "Unique_Values": df.nunique(dropna=True)
})

display(final_report)

print("Total duplicate rows:", df.duplicated().sum())


,Data_Type,Missing_Count,Missing_%,Unique_Values
Employee_ID,string,0,0.00,1020
First_Name,object,0,0.00,8
Last_Name,object,0,0.00,8
Age,Int64,0,0.00,4
Department_Region,object,0,0.00,36
Status,object,0,0.00,3
Join_Date,datetime64[ns],0,0.00,760
Salary,float64,0,0.00,979
Email,object,0,0.00,64
Phone,string,0,0.00,1020


Total duplicate rows: 0


## 11. Final Dataset Preview

The cleaned dataset is now ready for downstream analysis. Additional validation columns (`Phone_Length` and `Phone_Length_Status`) are retained to document contact-number quality rather than silently changing uncertain source data.


In [113]:
display(df.head(10))
print("\nFinal shape:", df.shape)


,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region,Phone_Length,Phone_Length_Status
0,EMP1000,Bob,Davis,25,DevOps-California,Active,2021-04-02,"59,767.65",bob.davis@example.com,1651623197,Average,True,Devops,California,10,Valid length
1,EMP1001,Bob,Brown,30,Finance-Texas,Active,2020-07-10,"65,304.66",bob.brown@example.com,1898471390,Excellent,True,Finance,Texas,10,Valid length
2,EMP1002,Alice,Jones,30,Admin-Nevada,Pending,2023-12-07,"88,145.90",alice.jones@example.com,5596363211,Good,True,Admin,Nevada,10,Valid length
3,EMP1003,Eva,Davis,25,Admin-Nevada,Inactive,2021-11-27,"69,450.99",eva.davis@example.com,3476490784,Good,True,Admin,Nevada,10,Valid length
4,EMP1004,Frank,Williams,25,Cloud Tech-Florida,Active,2022-01-05,"109,324.61",frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida,10,Valid length
5,EMP1005,Alice,Garcia,40,Sales-Texas,Inactive,2020-06-10,"88,642.84",alice.garcia@example.com,5409003485,Good,False,Sales,Texas,10,Valid length
6,EMP1006,Frank,Jones,30,Admin-Nevada,Active,2020-04-03,"96,288.43",frank.jones@example.com,4518376063,Good,False,Admin,Nevada,10,Valid length
7,EMP1007,Bob,Jones,30,Cloud Tech-Florida,Inactive,2022-07-17,"94,497.91",bob.jones@example.com,4134327559,Average,True,Cloud Tech,Florida,10,Valid length
8,EMP1008,Frank,Davis,35,Admin-Nevada,Inactive,2023-12-08,"115,565.82",frank.davis@example.com,4177656123,Excellent,True,Admin,Nevada,10,Valid length
9,EMP1009,Charlie,Johnson,30,DevOps-New York,Active,2022-08-04,"76,561.88",charlie.johnson@example.com,8156985699,Excellent,True,Devops,New York,10,Valid length



Final shape: (1020, 16)


## 12. Export Cleaned Dataset

The cleaned data is exported as a CSV file so it can be reused in Excel, Power BI, SQL workflows, or future Python analysis.


In [114]:
# Export the cleaned dataset
output_file = "cleaned_employee_dataset.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset exported successfully: {output_file}")


Cleaned dataset exported successfully: cleaned_employee_dataset.csv


# Conclusion

The employee dataset was systematically transformed into a cleaner, analysis-ready dataset.

### Key cleaning actions completed
1. Inspected the dataset structure, missing values, duplicates, data types, and validity/range issues.
2. Filled missing **Age** values using the median.
3. Filled missing **Salary** values using the median.
4. Checked for duplicate rows and removed any duplicates found.
5. Standardized names, status, performance categories, emails, and combined department/region text.
6. Converted **Join_Date** to datetime.
7. Converted **Employee_ID** and **Phone** to string identifiers and validated phone-number lengths without inventing unknown values.
8. Used the **IQR method** to detect potential outliers in Age and Salary; no IQR-based outliers required removal or capping.
9. Generated a final data quality report and exported the cleaned CSV.

### Final recommendation
The cleaned dataset can now be safely used for exploratory analysis, visualization, reporting, Excel/Power BI dashboards, and other analytical tasks. Phone records marked **Review** should be verified against the original source before being used for contact operations.

**thank you**
